In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
ПОЛНАЯ РЕАЛИЗАЦИЯ АЛГОРИТМА «КУЗНЕЧИК» (ГОСТ Р 34.12-2015)

Данная программа реализует блочный шифр «Кузнечик» согласно ГОСТ Р 34.12-2015
с поддержкой всех основных режимов шифрования и опциональной имитовставкой.

"""

import os
import sys
import hashlib
import getpass
import time
from typing import List, Optional, Union, Tuple
from enum import Enum


class EncryptionMode(Enum):
    """
    Перечисление доступных режимов шифрования.
    
    Поддерживаются следующие режимы:
    - ECB (Electronic Codebook) - режим простой замены
    - CBC (Cipher Block Chaining) - режим сцепления блоков
    - CFB (Cipher Feedback) - режим обратной связи по шифртексту
    - OFB (Output Feedback) - режим обратной связи по выходу
    - CTR (Counter) - режим счетчика
    """
    ECB = "ECB"
    CBC = "CBC"
    CFB = "CFB"
    OFB = "OFB"
    CTR = "CTR"


class Kuznechik:
    """
    Ядро алгоритма шифрования «Кузнечик» (ГОСТ Р 34.12-2015).
    
    Класс реализует все криптографические преобразования:
    - S-блок (нелинейное преобразование)
    - R-преобразование (линейное перемешивание)
    - L-преобразование (16-кратное применение R)
    - Генерацию раундовых ключей через сеть Фейстеля
    - Базовые операции шифрования/расшифрования блоков
    - Вычисление и проверку имитовставки (MAC)
    """
    
    # -------------------------------------------------------------------------
    # Таблицы замен (S-блоки) согласно ГОСТ Р 34.12-2015
    # -------------------------------------------------------------------------
    
    # Прямое нелинейное преобразование (Pi)
    PI = [
        0xFC, 0xEE, 0xDD, 0x11, 0xCF, 0x6E, 0x31, 0x16,
        0xFB, 0xC4, 0xFA, 0xDA, 0x23, 0xC5, 0x04, 0x4D,
        0xE9, 0x77, 0xF0, 0xDB, 0x93, 0x2E, 0x99, 0xBA,
        0x17, 0x36, 0xF1, 0xBB, 0x14, 0xCD, 0x5F, 0xC1,
        0xF9, 0x18, 0x65, 0x5A, 0xE2, 0x5C, 0xEF, 0x21,
        0x81, 0x1C, 0x3C, 0x42, 0x8B, 0x01, 0x8E, 0x4F,
        0x05, 0x84, 0x02, 0xAE, 0xE3, 0x6A, 0x8F, 0xA0,
        0x06, 0x0B, 0xED, 0x98, 0x7F, 0xD4, 0xD3, 0x1F,
        0xEB, 0x34, 0x2C, 0x51, 0xEA, 0xC8, 0x48, 0xAB,
        0xF2, 0x2A, 0x68, 0xA2, 0xFD, 0x3A, 0xCE, 0xCC,
        0xB5, 0x70, 0x0E, 0x56, 0x08, 0x0C, 0x76, 0x12,
        0xBF, 0x72, 0x13, 0x47, 0x9C, 0xB7, 0x5D, 0x87,
        0x15, 0xA1, 0x96, 0x29, 0x10, 0x7B, 0x9A, 0xC7,
        0xF3, 0x91, 0x78, 0x6F, 0x9D, 0x9E, 0xB2, 0xB1,
        0x32, 0x75, 0x19, 0x3D, 0xFF, 0x35, 0x8A, 0x7E,
        0x6D, 0x54, 0xC6, 0x80, 0xC3, 0xBD, 0x0D, 0x57,
        0xDF, 0xF5, 0x24, 0xA9, 0x3E, 0xA8, 0x43, 0xC9,
        0xD7, 0x79, 0xD6, 0xF6, 0x7C, 0x22, 0xB9, 0x03,
        0xE0, 0x0F, 0xEC, 0xDE, 0x7A, 0x94, 0xB0, 0xBC,
        0xDC, 0xE8, 0x28, 0x50, 0x4E, 0x33, 0x0A, 0x4A,
        0xA7, 0x97, 0x60, 0x73, 0x1E, 0x00, 0x62, 0x44,
        0x1A, 0xB8, 0x38, 0x82, 0x64, 0x9F, 0x26, 0x41,
        0xAD, 0x45, 0x46, 0x92, 0x27, 0x5E, 0x55, 0x2F,
        0x8C, 0xA3, 0xA5, 0x7D, 0x69, 0xD5, 0x95, 0x3B,
        0x07, 0x58, 0xB3, 0x40, 0x86, 0xAC, 0x1D, 0xF7,
        0x30, 0x37, 0x6B, 0xE4, 0x88, 0xD9, 0xE7, 0x89,
        0xE1, 0x1B, 0x83, 0x49, 0x4C, 0x3F, 0xF8, 0xFE,
        0x8D, 0x53, 0xAA, 0x90, 0xCA, 0xD8, 0x85, 0x61,
        0x20, 0x71, 0x67, 0xA4, 0x2D, 0x2B, 0x09, 0x5B,
        0xCB, 0x9B, 0x25, 0xD0, 0xBE, 0xE5, 0x6C, 0x52,
        0x59, 0xA6, 0x74, 0xD2, 0xE6, 0xF4, 0xB4, 0xC0,
        0xD1, 0x66, 0xAF, 0xC2, 0x39, 0x4B, 0x63, 0xB6
    ]
    
    # Обратное нелинейное преобразование (обратная таблица)
    REVERSE_PI = [
        165, 45, 50, 143, 14, 48, 56, 192, 84, 230, 158, 57, 85, 126, 82, 145,
        100, 3, 87, 90, 28, 96, 7, 24, 33, 114, 168, 209, 41, 198, 164, 63,
        224, 39, 141, 12, 130, 234, 174, 180, 154, 99, 73, 229, 66, 228, 21, 183,
        200, 6, 112, 157, 65, 117, 25, 201, 170, 252, 77, 191, 42, 115, 132, 213,
        195, 175, 43, 134, 167, 177, 178, 91, 70, 211, 159, 253, 212, 15, 156, 47,
        155, 67, 239, 217, 121, 182, 83, 127, 193, 240, 35, 231, 37, 94, 181, 30,
        162, 223, 166, 254, 172, 34, 249, 226, 74, 188, 53, 202, 238, 120, 5, 107,
        81, 225, 89, 163, 242, 113, 86, 17, 106, 137, 148, 101, 140, 187, 119, 60,
        123, 40, 171, 210, 49, 222, 196, 95, 204, 207, 118, 44, 184, 216, 46, 54,
        219, 105, 179, 20, 149, 190, 98, 161, 59, 22, 102, 233, 92, 108, 109, 173,
        55, 97, 75, 185, 227, 186, 241, 160, 133, 131, 218, 71, 197, 176, 51, 250,
        150, 111, 110, 194, 246, 80, 255, 93, 169, 142, 23, 27, 151, 125, 236, 88,
        247, 31, 251, 124, 9, 13, 122, 103, 69, 135, 220, 232, 79, 29, 78, 4,
        235, 248, 243, 62, 61, 189, 138, 136, 221, 205, 11, 19, 152, 2, 147, 128,
        144, 208, 36, 52, 203, 237, 244, 206, 153, 16, 68, 64, 146, 58, 1, 38,
        18, 26, 72, 104, 245, 129, 139, 199, 214, 32, 10, 8, 0, 76, 215, 116
    ]
    
    # Вектор линейного преобразования
    L_VECTOR = [148, 32, 133, 16, 194, 192, 1, 251, 1, 192, 194, 16, 133, 32, 148, 1]
    
    # Порождающий полином
    POLY = 0xC3
    BLOCK_SIZE = 16
    
    def __init__(self):
        self.iter_c = [None] * 32
        self.iter_k = [None] * 10
        self.encrypt_count = 0
        self.decrypt_count = 0
    
    # -------------------------------------------------------------------------
    # Вспомогательные криптографические функции
    # -------------------------------------------------------------------------
    
    def _sha512(self, data: bytes) -> bytes:
        """SHA-512 хеш функция (первые 32 байта)"""
        return hashlib.sha512(data).digest()[:32]
    
    def _length_to_32_bytes(self, key_str: str) -> bytes:
        """Преобразование строки в 32-байтовый ключ (50 итераций SHA-512)"""
        res = self._sha512(key_str.encode())
        for _ in range(50):
            res = self._sha512(res)
        return res
    
    def _galois_multiply(self, a: int, b: int) -> int:
        """Умножение в поле Галуа GF(2^8)"""
        p = 0
        counter = 0
        a_val = a & 0xFF
        b_val = b & 0xFF
        
        while counter < 8 and a_val != 0 and b_val != 0:
            if b_val & 1:
                p ^= a_val
            hi_bit = a_val & 0x80
            a_val = (a_val << 1) & 0xFF
            if hi_bit:
                a_val ^= self.POLY
            b_val >>= 1
            counter += 1
        
        return p & 0xFF
    
    def _xor(self, a: bytes, b: bytes) -> bytes:
        """XOR двух массивов одинаковой длины"""
        return bytes([a[i] ^ b[i] for i in range(len(a))])
    
    def _xor_lists(self, a: List[int], b: List[int]) -> List[int]:
        """XOR двух списков"""
        return [a[i] ^ b[i] for i in range(len(a))]
    
    # -------------------------------------------------------------------------
    # Основные криптографические преобразования
    # -------------------------------------------------------------------------
    
    def _s_box_substitution(self, data: List[int]) -> List[int]:
        """Нелинейное преобразование S"""
        return [self.PI[b] for b in data]
    
    def _s_box_substitution_reverse(self, data: List[int]) -> List[int]:
        """Обратное нелинейное преобразование"""
        return [self.REVERSE_PI[b] for b in data]
    
    def _r_transformation(self, data: List[int]) -> List[int]:
        """Преобразование R"""
        a_zero = 0
        for i in range(16):
            a_zero ^= self._galois_multiply(data[i], self.L_VECTOR[i])
        
        state = [0] * 16
        for i in range(15, 0, -1):
            state[i] = data[i - 1]
        state[0] = a_zero
        
        return state
    
    def _r_transformation_reverse(self, data: List[int]) -> List[int]:
        """Обратное преобразование R"""
        a_last = data[0]
        state = [0] * 16
        
        for i in range(15):
            state[i] = data[i + 1]
        
        for i in range(15, -1, -1):
            a_last ^= self._galois_multiply(state[i], self.L_VECTOR[i])
        
        state[15] = a_last
        return state
    
    def _l_transformation(self, data: List[int]) -> List[int]:
        """Линейное преобразование L"""
        state = data.copy()
        for _ in range(16):
            state = self._r_transformation(state)
        return state
    
    def _l_transformation_reverse(self, data: List[int]) -> List[int]:
        """Обратное линейное преобразование"""
        state = data.copy()
        for _ in range(16):
            state = self._r_transformation_reverse(state)
        return state
    
    # -------------------------------------------------------------------------
    # Генерация ключей
    # -------------------------------------------------------------------------
    
    def _feistel_network(self, first_key: List[int], second_key: List[int], 
                         iter_const: List[int]) -> Tuple[List[int], List[int]]:
        """Преобразование ячейки Фейстеля"""
        state = self._xor_lists(first_key, iter_const)
        state = self._s_box_substitution(state)
        state = self._l_transformation(state)
        return (self._xor_lists(state, second_key), first_key.copy())
    
    def _expand_keys(self, key: bytes):
        """Развертка ключа - генерация 10 раундовых ключей"""
        key_list = list(key)
        
        # Генерация итерационных констант
        for i in range(32):
            iter_num = [0] * 16
            iter_num[15] = i + 1
            self.iter_c[i] = self._l_transformation(iter_num)
        
        # Разделение ключа
        a = key_list[:16]
        b = key_list[16:]
        
        # Первые два ключа
        self.iter_k[0] = b
        self.iter_k[1] = a
        
        # Генерация остальных ключей
        for i in range(4):
            for j in range(8):
                idx = 8 * i + j
                tmp = self._feistel_network(a, b, self.iter_c[idx])
                a = tmp[1]
                b = tmp[0]
            self.iter_k[2 * i + 2] = a
            self.iter_k[2 * i + 3] = b
    
    # -------------------------------------------------------------------------
    # Управление ключами
    # -------------------------------------------------------------------------
    
    def set_key_from_string(self, key_str: str) -> bytes:
        """Установка ключа из строки (пароля)"""
        key_bytes = self._length_to_32_bytes(key_str)
        self._expand_keys(key_bytes)
        return key_bytes
    
    def set_key_from_bytes(self, key_bytes: bytes):
        """Установка ключа из байтов (32 байта)"""
        if len(key_bytes) != 32:
            raise ValueError(f"Ключ должен быть 32 байта, получено {len(key_bytes)}")
        self._expand_keys(key_bytes)
    
    # -------------------------------------------------------------------------
    # Дополнение данных (padding)
    # -------------------------------------------------------------------------
    
    def _pad_data(self, data: bytes) -> bytes:
        """
        Дополнение данных до размера блока по схеме из Kotlin.
        """
        if len(data) % 16 == 0:
            return data
        
        num_blocks = len(data) // 16 + 1
        number_of_null = num_blocks * 16 - len(data)
        padded = bytearray(data) + bytearray(number_of_null)
        
        if number_of_null == 1:
            padded[-1] = 0x80
        else:
            for i in range(len(padded) - 1, -1, -1):
                if i == len(padded) - 1:
                    padded[i] = 0x81
                elif padded[i] != 0:
                    padded[i + 1] = 0x01
                    break
        
        return bytes(padded)
    
    def _unpad_data(self, data: bytes) -> bytes:
        """
        Удаление дополнения из данных.
        """
        if len(data) == 0:
            return data
        
        result = bytearray(data)
        
        if result[-1] == 0x80:
            result = result[:-1]
        elif result[-1] == 0x81:
            zero_count = 0
            for i in range(len(result) - 1, -1, -1):
                if result[i] in (0x81, 0x01, 0):
                    zero_count += 1
                else:
                    break
            if zero_count > 0:
                result = result[:-zero_count]
        
        return bytes(result)
    
    # -------------------------------------------------------------------------
    # Базовые операции с блоками
    # -------------------------------------------------------------------------
    
    def encrypt_block(self, block: bytes) -> bytes:
        """Шифрование одного блока"""
        if self.iter_k[0] is None:
            raise RuntimeError("Ключ не установлен")
        
        state = list(block)
        
        for j in range(9):
            state = self._xor_lists(state, self.iter_k[j])
            state = self._s_box_substitution(state)
            state = self._l_transformation(state)
        
        state = self._xor_lists(state, self.iter_k[9])
        
        self.encrypt_count += 1
        return bytes(state)
    
    def decrypt_block(self, block: bytes) -> bytes:
        """Расшифрование одного блока"""
        if self.iter_k[0] is None:
            raise RuntimeError("Ключ не установлен")
        
        state = list(block)
        
        state = self._xor_lists(state, self.iter_k[9])
        
        for j in range(8, -1, -1):
            state = self._l_transformation_reverse(state)
            state = self._s_box_substitution_reverse(state)
            state = self._xor_lists(state, self.iter_k[j])
        
        self.decrypt_count += 1
        return bytes(state)
    
    # -------------------------------------------------------------------------
    # Режимы шифрования
    # -------------------------------------------------------------------------
    
    def encrypt_ecb(self, data: bytes, verbose: bool = False) -> bytes:
        """Режим ECB"""
        if verbose:
            print(f"\n📋 ECB Encryption:")
            print(f"   Original data ({len(data)} bytes): {data.hex()}")
        
        padded = self._pad_data(data)
        if verbose:
            print(f"   Padded data ({len(padded)} bytes): {padded.hex()}")
        
        result = bytearray()
        for i in range(0, len(padded), 16):
            block = padded[i:i+16]
            encrypted = self.encrypt_block(block)
            result.extend(encrypted)
            if verbose:
                print(f"   Block {i//16}: {block.hex()} -> {encrypted.hex()}")
        
        return bytes(result)
    
    def decrypt_ecb(self, data: bytes, verbose: bool = False) -> bytes:
        """Расшифрование ECB"""
        if len(data) % 16 != 0:
            raise ValueError("Размер данных должен быть кратен 16")
        
        if verbose:
            print(f"\n📋 ECB Decryption:")
            print(f"   Encrypted data ({len(data)} bytes): {data.hex()}")
        
        result = bytearray()
        for i in range(0, len(data), 16):
            block = data[i:i+16]
            decrypted = self.decrypt_block(block)
            result.extend(decrypted)
            if verbose:
                print(f"   Block {i//16}: {block.hex()} -> {decrypted.hex()}")
        
        unpadded = self._unpad_data(bytes(result))
        if verbose:
            print(f"   Unpadded data ({len(unpadded)} bytes): {unpadded.hex()}")
        
        return unpadded
    
    def encrypt_cbc(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Режим CBC"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        
        if verbose:
            print(f"\n📋 CBC Encryption:")
            print(f"   Original data ({len(data)} bytes): {data.hex()}")
            print(f"   IV: {iv.hex()}")
        
        padded = self._pad_data(data)
        if verbose:
            print(f"   Padded data ({len(padded)} bytes): {padded.hex()}")
        
        result = bytearray()
        prev = iv
        for i in range(0, len(padded), 16):
            block = padded[i:i+16]
            xored = bytes([block[j] ^ prev[j] for j in range(16)])
            encrypted = self.encrypt_block(xored)
            result.extend(encrypted)
            if verbose:
                print(f"   Block {i//16}: {block.hex()} XOR {prev.hex()} = {xored.hex()} -> {encrypted.hex()}")
            prev = encrypted
        
        return bytes(result)
    
    def decrypt_cbc(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Расшифрование CBC"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        if len(data) % 16 != 0:
            raise ValueError("Размер данных должен быть кратен 16")
        
        if verbose:
            print(f"\n📋 CBC Decryption:")
            print(f"   Encrypted data ({len(data)} bytes): {data.hex()}")
            print(f"   IV: {iv.hex()}")
        
        result = bytearray()
        prev = iv
        for i in range(0, len(data), 16):
            block = data[i:i+16]
            decrypted = self.decrypt_block(block)
            plain = bytes([decrypted[j] ^ prev[j] for j in range(16)])
            result.extend(plain)
            if verbose:
                print(f"   Block {i//16}: {block.hex()} -> {decrypted.hex()} XOR {prev.hex()} = {plain.hex()}")
            prev = block
        
        unpadded = self._unpad_data(bytes(result))
        if verbose:
            print(f"   Unpadded data ({len(unpadded)} bytes): {unpadded.hex()}")
        
        return unpadded
    
    def encrypt_cfb(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Режим CFB"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        
        if verbose:
            print(f"\n📋 CFB Encryption:")
            print(f"   Original data ({len(data)} bytes): {data.hex()}")
            print(f"   IV: {iv.hex()}")
        
        result = bytearray()
        feedback = bytearray(iv)
        
        for i in range(0, len(data), 16):
            encrypted_feedback = self.encrypt_block(bytes(feedback))
            if verbose:
                print(f"   Feedback {i//16}: {feedback.hex()} -> encrypted: {encrypted_feedback.hex()}")
            
            block = data[i:i+16]
            for j in range(len(block)):
                result.append(block[j] ^ encrypted_feedback[j])
            
            if verbose:
                print(f"   Block {i//16}: {block.hex()} XOR {encrypted_feedback.hex()[:len(block)*2]} = {result[-len(block):].hex()}")
            
            if len(block) == 16:
                feedback = result[-16:]
            else:
                for j in range(len(block)):
                    feedback[j] = result[-len(block) + j]
        
        return bytes(result)
    
    def decrypt_cfb(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Расшифрование CFB"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        
        if verbose:
            print(f"\n📋 CFB Decryption:")
            print(f"   Encrypted data ({len(data)} bytes): {data.hex()}")
            print(f"   IV: {iv.hex()}")
        
        result = bytearray()
        feedback = bytearray(iv)
        
        for i in range(0, len(data), 16):
            encrypted_feedback = self.encrypt_block(bytes(feedback))
            if verbose:
                print(f"   Feedback {i//16}: {feedback.hex()} -> encrypted: {encrypted_feedback.hex()}")
            
            block = data[i:i+16]
            cipher_block = block
            
            for j in range(len(block)):
                result.append(block[j] ^ encrypted_feedback[j])
            
            if verbose:
                print(f"   Block {i//16}: {block.hex()} XOR {encrypted_feedback.hex()[:len(block)*2]} = {result[-len(block):].hex()}")
            
            if len(block) == 16:
                feedback = bytearray(cipher_block)
            else:
                for j in range(len(block)):
                    feedback[j] = cipher_block[j]
        
        return bytes(result)
    
    def encrypt_ofb(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Режим OFB"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        
        if verbose:
            print(f"\n📋 OFB Encryption:")
            print(f"   Original data ({len(data)} bytes): {data.hex()}")
            print(f"   IV: {iv.hex()}")
        
        result = bytearray()
        state = bytearray(iv)
        
        for i in range(0, len(data), 16):
            encrypted_state = self.encrypt_block(bytes(state))
            if verbose:
                print(f"   State {i//16}: {state.hex()} -> encrypted: {encrypted_state.hex()}")
            
            state = bytearray(encrypted_state)
            
            block = data[i:i+16]
            for j in range(len(block)):
                result.append(block[j] ^ encrypted_state[j])
            
            if verbose:
                print(f"   Block {i//16}: {block.hex()} XOR {encrypted_state.hex()[:len(block)*2]} = {result[-len(block):].hex()}")
        
        return bytes(result)
    
    def decrypt_ofb(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Расшифрование OFB"""
        return self.encrypt_ofb(data, iv, verbose)
    
    def encrypt_ctr(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Режим CTR"""
        if len(iv) != 16:
            raise ValueError("IV должен быть 16 байт")
        
        if verbose:
            print(f"\n📋 CTR Encryption:")
            print(f"   Original data ({len(data)} bytes): {data.hex()}")
            print(f"   Initial counter: {iv.hex()}")
        
        result = bytearray()
        counter = bytearray(iv)
        
        for i in range(0, len(data), 16):
            encrypted_counter = self.encrypt_block(bytes(counter))
            if verbose:
                print(f"   Counter {i//16}: {counter.hex()} -> encrypted: {encrypted_counter.hex()}")
            
            block = data[i:i+16]
            for j in range(len(block)):
                result.append(block[j] ^ encrypted_counter[j])
            
            if verbose:
                print(f"   Block {i//16}: {block.hex()} XOR {encrypted_counter.hex()[:len(block)*2]} = {result[-len(block):].hex()}")
            
            for j in range(15, -1, -1):
                counter[j] = (counter[j] + 1) & 0xFF
                if counter[j] != 0:
                    break
        
        return bytes(result)
    
    def decrypt_ctr(self, data: bytes, iv: bytes, verbose: bool = False) -> bytes:
        """Расшифрование CTR"""
        return self.encrypt_ctr(data, iv, verbose)
    
    # -------------------------------------------------------------------------
    # Имитовставка (MAC)
    # -------------------------------------------------------------------------
    
    def _generate_mac_keys(self) -> Tuple[bytearray, bytearray]:
        """
        Генерация подключей K1 и K2 для имитовставки.
        
        Returns:
            Кортеж (K1, K2) - подключи для MAC
        """
        # K1 = E(0)
        k1 = bytearray(16)
        k1 = bytearray(self.encrypt_block(bytes(k1)))
        
        # Сдвиг K1 влево
        carry = 0
        for i in range(15, -1, -1):
            new_carry = 1 if k1[i] >= 0x80 else 0
            k1[i] = ((k1[i] << 1) | carry) & 0xFF
            carry = new_carry
        if carry:
            k1[15] ^= 0x87
        
        # K2 = сдвиг K1
        k2 = bytearray(k1)
        carry = 0
        for i in range(15, -1, -1):
            new_carry = 1 if k2[i] >= 0x80 else 0
            k2[i] = ((k2[i] << 1) | carry) & 0xFF
            carry = new_carry
        if carry:
            k2[15] ^= 0x87
        
        return k1, k2
    
    def compute_mac(self, data: bytes, mac_len: int = 8, verbose: bool = False) -> bytes:
        """
        Вычисление имитовставки (MAC) для данных.
        
        Args:
            data: Данные для вычисления имитовставки
            mac_len: Длина имитовставки в байтах (обычно 4 или 8)
            verbose: Если True, выводить отладочную информацию
            
        Returns:
            Имитовставка длиной mac_len байт
        """
        if self.iter_k[0] is None:
            raise RuntimeError("Ключ не установлен")
        
        if verbose:
            print(f"\n🔐 MAC Computation (len={mac_len}):")
            print(f"   Data ({len(data)} bytes): {data.hex()}")
        
        # Генерация подключей
        k1, k2 = self._generate_mac_keys()
        if verbose:
            print(f"   K1: {k1.hex()}")
            print(f"   K2: {k2.hex()}")
        
        # Дополнение данных
        if len(data) % 16 == 0:
            padded = data
            is_added = False
            if verbose:
                print(f"   No padding needed")
        else:
            padded = data + b'\x80' + b'\x00' * (15 - (len(data) % 16))
            is_added = True
            if verbose:
                print(f"   Padded data: {padded.hex()}")
        
        # Разбиваем на блоки
        blocks = [padded[i:i+16] for i in range(0, len(padded), 16)]
        if verbose:
            print(f"   Number of blocks: {len(blocks)}")
        
        if not blocks:
            return bytes(mac_len)
        
        # Начальное состояние
        state = bytearray(self.encrypt_block(blocks[0]))
        if verbose:
            print(f"   Block 0: {blocks[0].hex()} -> state: {state.hex()}")
        
        # Промежуточные блоки
        for idx in range(1, len(blocks) - 1):
            block = blocks[idx]
            for j in range(16):
                state[j] ^= block[j]
            if verbose:
                print(f"   After XOR with block {idx}: {state.hex()}")
            state = bytearray(self.encrypt_block(bytes(state)))
            if verbose:
                print(f"   After encryption: {state.hex()}")
        
        # Последний блок
        last = blocks[-1]
        key = k2 if is_added else k1
        if verbose:
            print(f"   Last block: {last.hex()}")
            print(f"   Using key: {key.hex()}")
        
        for j in range(16):
            state[j] ^= last[j] ^ key[j]
        if verbose:
            print(f"   After XOR with last block and key: {state.hex()}")
        
        # Финальное шифрование
        state = bytearray(self.encrypt_block(bytes(state)))
        if verbose:
            print(f"   Final encryption: {state.hex()}")
        
        mac = bytes(state[:mac_len])
        if verbose:
            print(f"   MAC ({mac_len} bytes): {mac.hex()}")
        
        return mac
    
    def verify_mac(self, data: bytes, mac: bytes, verbose: bool = False) -> bool:
        """
        Проверка имитовставки.
        
        Args:
            data: Данные для проверки
            mac: Ожидаемая имитовставка
            verbose: Если True, выводить отладочную информацию
            
        Returns:
            True, если имитовставка совпадает, иначе False
        """
        computed = self.compute_mac(data, len(mac), verbose)
        result = computed == mac
        if verbose:
            print(f"\n🔐 MAC Verification:")
            print(f"   Expected: {mac.hex()}")
            print(f"   Computed: {computed.hex()}")
            print(f"   Result: {'✅ MATCH' if result else '❌ MISMATCH'}")
        return result


# ==================== УПРАВЛЕНИЕ КЛЮЧАМИ ====================

class KeyManager:
    """Класс для управления ключами шифрования."""
    
    KEY_DIR = "keys"
    
    def __init__(self):
        if not os.path.exists(self.KEY_DIR):
            os.makedirs(self.KEY_DIR)
    
    def save_key(self, name: str, key: bytes) -> str:
        """Сохранение ключа в файл."""
        clean_name = "".join(c for c in name if c.isalnum() or c in "._- ")
        clean_name = clean_name.replace(" ", "_")
        
        filename = f"{clean_name}.key"
        filepath = os.path.join(self.KEY_DIR, filename)
        
        with open(filepath, 'wb') as f:
            f.write(key)
        
        return filepath
    
    def load_key(self, path: str) -> bytes:
        """Загрузка ключа из файла."""
        if not os.path.isabs(path) and not os.path.exists(path):
            test_path = os.path.join(self.KEY_DIR, path)
            if os.path.exists(test_path):
                path = test_path
        
        if not os.path.exists(path):
            raise FileNotFoundError(f"Файл ключа не найден: {path}")
        
        with open(path, 'rb') as f:
            data = f.read()
        
        if len(data) == 32:
            return data
        elif len(data) == 48:
            print("⚠️ Обнаружен ключ старого формата (с солью). Используется только ключ (первые 32 байта).")
            return data[:32]
        else:
            raise ValueError(f"Неверный размер ключа: {len(data)} байт")
    
    def list_keys(self) -> List[dict]:
        """Получение списка сохраненных ключей."""
        keys = []
        for f in os.listdir(self.KEY_DIR):
            if f.endswith('.key'):
                filepath = os.path.join(self.KEY_DIR, f)
                stat = os.stat(filepath)
                keys.append({
                    'filename': f,
                    'path': filepath,
                    'size': stat.st_size,
                    'modified': time.ctime(stat.st_mtime)
                })
        return keys


# ==================== ИНТЕРАКТИВНОЕ МЕНЮ ====================

class InteractiveMenu:
    """
    Интерактивное консольное меню для работы с программой.
    """
    
    MODE_DESCRIPTIONS = {
        "ECB": "ECB  - Electronic Codebook (простая замена)",
        "CBC": "CBC  - Cipher Block Chaining (сцепление блоков) - требует IV 16 байт",
        "CFB": "CFB  - Cipher Feedback (обратная связь по шифртексту) - требует IV 16 байт",
        "OFB": "OFB  - Output Feedback (обратная связь по выходу) - требует IV 16 байт",
        "CTR": "CTR  - Counter (режим счетчика) - требует IV 16 байт",
    }
    
    def __init__(self):
        self.cipher = Kuznechik()
        self.key_manager = KeyManager()
        self.current_key = None
        self.current_key_name = None
        self.current_mode = None
        self.current_gamma = None
        self.use_mac = False          # Флаг использования имитовставки
        self.mac_len = 8              # Длина имитовставки (по умолчанию 8 байт)
    
    def clear(self):
        """Очистка экрана."""
        os.system('cls' if os.name == 'nt' else 'clear')
    
    def print_header(self):
        """Вывод заголовка программы."""
        print("=" * 70)
        print("🔐 КУЗНЕЧИК - ШИФРОВАНИЕ ПО ГОСТ Р 34.12-2015")
        print("=" * 70)
        print()
    
    def print_key_status(self):
        """Вывод статуса текущего ключа и режима."""
        if self.current_key:
            key_hash = hashlib.sha256(self.current_key).hexdigest()
            print(f"🔑 Текущий ключ: {self.current_key_name} [{key_hash[:16]}...]")
        else:
            print(f"❌ Ключ не установлен")
        if self.current_mode:
            print(f"📋 Текущий режим: {self.current_mode}")
        if self.current_gamma:
            print(f"🎲 Текущая синхропосылка: {self.current_gamma.hex()}")
        print(f"🛡️  Имитовставка: {'ВКЛ' if self.use_mac else 'ВЫКЛ'} (длина: {self.mac_len} байт)")
        print()
    
    def run(self):
        """Запуск главного меню."""
        while True:
            self.clear()
            self.print_header()
            self.print_key_status()
            
            print("ГЛАВНОЕ МЕНЮ:")
            print("  [1] 🔐 Зашифровать файл")
            print("  [2] 🔓 Расшифровать файл")
            print("  [3] 📋 Выбрать режим шифрования")
            print("  [4] 🛡️  Настройки имитовставки (MAC)")
            print("  [5] 🔑 Управление ключами")
            print("  [6] 🎲 Установить синхропосылку (IV)")
            print("  [7] 📊 Статистика")
            print("  [8] 🧪 Тестовый цикл")
            print("  [0] 🚪 Выход")
            print()
            
            choice = input("Выберите действие: ").strip()
            
            if choice == '1':
                self.encrypt_menu()
            elif choice == '2':
                self.decrypt_menu()
            elif choice == '3':
                self.select_mode_menu()
            elif choice == '4':
                self.mac_settings_menu()
            elif choice == '5':
                self.key_menu()
            elif choice == '6':
                self.set_gamma_menu()
            elif choice == '7':
                self.stats_menu()
            elif choice == '8':
                self.test_menu()
            elif choice == '0':
                print("\n👋 До свидания!")
                break
    
    def select_mode_menu(self):
        """Меню выбора режима шифрования."""
        self.clear()
        self.print_header()
        print("📋 ВЫБОР РЕЖИМА ШИФРОВАНИЯ")
        print("-" * 70)
        
        modes = ["ECB", "CBC", "CFB", "OFB", "CTR"]
        for i, mode in enumerate(modes, 1):
            print(f"  [{i}] {self.MODE_DESCRIPTIONS[mode]}")
        print()
        
        choice = input("Выберите режим (1-5): ").strip()
        if choice.isdigit():
            idx = int(choice) - 1
            if 0 <= idx < len(modes):
                self.current_mode = modes[idx]
                print(f"\n✅ Выбран режим: {self.current_mode}")
        
        input("\nНажмите Enter...")
    
    def mac_settings_menu(self):
        """Меню настройки имитовставки."""
        self.clear()
        self.print_header()
        print("🛡️  НАСТРОЙКИ ИМИТОВСТАВКИ (MAC)")
        print("-" * 70)
        
        print(f"Текущее состояние: {'ВКЛ' if self.use_mac else 'ВЫКЛ'}")
        print(f"Длина имитовставки: {self.mac_len} байт")
        print()
        print("  [1] Включить имитовставку")
        print("  [2] Выключить имитовставку")
        print("  [3] Установить длину имитовставки (4 или 8 байт)")
        print("  [0] ↩️ Назад")
        print()
        
        choice = input("Выберите действие: ").strip()
        
        if choice == '1':
            self.use_mac = True
            print("✅ Имитовставка включена")
        elif choice == '2':
            self.use_mac = False
            print("✅ Имитовставка выключена")
        elif choice == '3':
            len_input = input("Введите длину имитовставки (4 или 8): ").strip()
            if len_input in ('4', '8'):
                self.mac_len = int(len_input)
                print(f"✅ Длина имитовставки установлена: {self.mac_len} байт")
            else:
                print("❌ Длина должна быть 4 или 8 байт")
        
        input("\nНажмите Enter...")
    
    def set_gamma_menu(self):
        """Меню установки синхропосылки (IV)."""
        self.clear()
        self.print_header()
        print("🎲 УСТАНОВКА СИНХРОПОСЫЛКИ (IV)")
        print("-" * 70)
        
        print("Источник синхропосылки:")
        print("  [1] Сгенерировать случайно (16 байт)")
        print("  [2] Ввести вручную (hex, 32 символа)")
        print("  [3] Загрузить из файла")
        print("  [4] Очистить (не использовать)")
        print()
        
        choice = input("Выберите источник: ").strip()
        
        if choice == '1':
            self.current_gamma = os.urandom(16)
            print(f"✅ Сгенерирована синхропосылка: {self.current_gamma.hex()}")
        
        elif choice == '2':
            gamma_hex = input("Введите IV в hex (32 символа): ").strip()
            try:
                gamma = bytes.fromhex(gamma_hex)
                if len(gamma) != 16:
                    print("❌ Синхропосылка должна быть 16 байт")
                else:
                    self.current_gamma = gamma
                    print(f"✅ Установлена синхропосылка: {self.current_gamma.hex()}")
            except ValueError:
                print("❌ Неверный hex формат")
        
        elif choice == '3':
            gamma_file = input("Путь к файлу с синхропосылкой: ").strip()
            if os.path.exists(gamma_file):
                with open(gamma_file, 'rb') as f:
                    gamma = f.read()
                if len(gamma) == 16:
                    self.current_gamma = gamma
                    print(f"✅ Загружена синхропосылка: {self.current_gamma.hex()}")
                else:
                    print("❌ Файл должен содержать 16 байт")
            else:
                print("❌ Файл не найден")
        
        elif choice == '4':
            self.current_gamma = None
            print("✅ Синхропосылка очищена")
        
        input("\nНажмите Enter...")
    
    def encrypt_menu(self):
        """Меню шифрования файла."""
        if not self.current_key:
            print("\n❌ Сначала установите ключ!")
            input("Нажмите Enter...")
            return
        
        if not self.current_mode:
            print("\n❌ Сначала выберите режим шифрования!")
            input("Нажмите Enter...")
            return
        
        need_gamma = ["CBC", "CFB", "OFB", "CTR"]
        if self.current_mode in need_gamma and not self.current_gamma:
            print(f"\n❌ Для режима {self.current_mode} требуется синхропосылка (IV)!")
            input("Нажмите Enter...")
            return
        
        self.clear()
        self.print_header()
        print(f"🔐 РЕЖИМ ШИФРОВАНИЯ: {self.current_mode}")
        print(f"🛡️  Имитовставка: {'ВКЛ' if self.use_mac else 'ВЫКЛ'}")
        print("-" * 70)
        
        in_file = input("Входной файл: ").strip()
        if not os.path.exists(in_file):
            print(f"❌ Файл {in_file} не найден")
            input("Нажмите Enter...")
            return
        
        out_file = input("Выходной файл: ").strip()
        
        # Спрашиваем, нужен ли подробный вывод
        verbose = input("\nПоказать подробный процесс? (y/n) [n]: ").lower() == 'y'
        
        print("\n" + "=" * 70)
        print("ПАРАМЕТРЫ ШИФРОВАНИЯ:")
        print(f"  Входной файл: {in_file}")
        print(f"  Выходной файл: {out_file}")
        print(f"  Режим: {self.current_mode}")
        if self.current_gamma:
            print(f"  IV: {self.current_gamma.hex()}")
        print(f"  Имитовставка: {'да' if self.use_mac else 'нет'}")
        if self.use_mac:
            print(f"  Длина MAC: {self.mac_len} байт")
        print(f"  Подробный вывод: {'да' if verbose else 'нет'}")
        print("=" * 70)
        
        if input("\nНачать шифрование? (y/n) [y]: ").lower() == 'n':
            return
        
        try:
            with open(in_file, 'rb') as f:
                data = f.read()
            
            print(f"\n📄 Прочитано {len(data)} байт")
            print(f"   Данные (hex): {data.hex()}")
            
            # Шифрование данных
            if self.current_mode == "ECB":
                encrypted = self.cipher.encrypt_ecb(data, verbose)
            elif self.current_mode == "CBC":
                encrypted = self.cipher.encrypt_cbc(data, self.current_gamma, verbose)
            elif self.current_mode == "CFB":
                encrypted = self.cipher.encrypt_cfb(data, self.current_gamma, verbose)
            elif self.current_mode == "OFB":
                encrypted = self.cipher.encrypt_ofb(data, self.current_gamma, verbose)
            elif self.current_mode == "CTR":
                encrypted = self.cipher.encrypt_ctr(data, self.current_gamma, verbose)
            else:
                raise ValueError(f"Неизвестный режим: {self.current_mode}")
            
            # Если включена имитовставка, вычисляем её и добавляем в конец файла
            mac = None
            if self.use_mac:
                print(f"\n🛡️  Вычисление имитовставки...")
                mac = self.cipher.compute_mac(data, self.mac_len, verbose)
                print(f"   Имитовставка: {mac.hex()}")
                
                # Добавляем имитовставку в конец зашифрованных данных
                final_data = encrypted + mac
            else:
                final_data = encrypted
            
            # Сохраняем результат
            with open(out_file, 'wb') as f:
                f.write(final_data)
            
            print(f"\n💾 Сохранено {len(final_data)} байт в {out_file}")
            if self.use_mac:
                print(f"   Зашифрованные данные: {len(encrypted)} байт")
                print(f"   Имитовставка: {len(mac)} байт")
            else:
                print(f"   Зашифрованные данные (hex): {encrypted.hex()}")
            
            print("\n✅ ШИФРОВАНИЕ ЗАВЕРШЕНО!")
            
        except Exception as e:
            print(f"\n❌ Ошибка: {e}")
        
        input("\nНажмите Enter...")
    
    def decrypt_menu(self):
        """Меню расшифрования файла."""
        if not self.current_key:
            print("\n❌ Сначала установите ключ!")
            input("Нажмите Enter...")
            return
        
        if not self.current_mode:
            print("\n❌ Сначала выберите режим шифрования!")
            input("Нажмите Enter...")
            return
        
        need_gamma = ["CBC", "CFB", "OFB", "CTR"]
        if self.current_mode in need_gamma and not self.current_gamma:
            print(f"\n❌ Для режима {self.current_mode} требуется синхропосылка (IV)!")
            input("Нажмите Enter...")
            return
        
        self.clear()
        self.print_header()
        print(f"🔓 РЕЖИМ РАСШИФРОВАНИЯ: {self.current_mode}")
        print(f"🛡️  Имитовставка: {'ВКЛ' if self.use_mac else 'ВЫКЛ'}")
        print("-" * 70)
        
        in_file = input("Входной файл (зашифрованный): ").strip()
        if not os.path.exists(in_file):
            print(f"❌ Файл {in_file} не найден")
            input("Нажмите Enter...")
            return
        
        out_file = input("Выходной файл (расшифрованный): ").strip()
        
        # Спрашиваем, нужен ли подробный вывод
        verbose = input("\nПоказать подробный процесс? (y/n) [n]: ").lower() == 'y'
        
        print("\n" + "=" * 70)
        print("ПАРАМЕТРЫ РАСШИФРОВАНИЯ:")
        print(f"  Входной файл: {in_file}")
        print(f"  Выходной файл: {out_file}")
        print(f"  Режим: {self.current_mode}")
        if self.current_gamma:
            print(f"  IV: {self.current_gamma.hex()}")
        print(f"  Имитовставка: {'да' if self.use_mac else 'нет'}")
        if self.use_mac:
            print(f"  Длина MAC: {self.mac_len} байт")
        print(f"  Подробный вывод: {'да' if verbose else 'нет'}")
        print("=" * 70)
        
        if input("\nНачать расшифрование? (y/n) [y]: ").lower() == 'n':
            return
        
        try:
            with open(in_file, 'rb') as f:
                file_data = f.read()
            
            print(f"\n📄 Прочитано {len(file_data)} байт")
            
            # Если включена имитовставка, отделяем её от зашифрованных данных
            if self.use_mac:
                if len(file_data) < self.mac_len:
                    raise ValueError(f"Файл слишком мал: {len(file_data)} байт, ожидается минимум {self.mac_len} байт для MAC")
                
                encrypted = file_data[:-self.mac_len]
                expected_mac = file_data[-self.mac_len:]
                
                print(f"   Зашифрованные данные: {len(encrypted)} байт")
                print(f"   Ожидаемая имитовставка: {expected_mac.hex()}")
            else:
                encrypted = file_data
            
            print(f"   Зашифрованные данные (hex): {encrypted.hex()}")
            
            # Расшифрование данных
            if self.current_mode == "ECB":
                decrypted = self.cipher.decrypt_ecb(encrypted, verbose)
            elif self.current_mode == "CBC":
                decrypted = self.cipher.decrypt_cbc(encrypted, self.current_gamma, verbose)
            elif self.current_mode == "CFB":
                decrypted = self.cipher.decrypt_cfb(encrypted, self.current_gamma, verbose)
            elif self.current_mode == "OFB":
                decrypted = self.cipher.decrypt_ofb(encrypted, self.current_gamma, verbose)
            elif self.current_mode == "CTR":
                decrypted = self.cipher.decrypt_ctr(encrypted, self.current_gamma, verbose)
            else:
                raise ValueError(f"Неизвестный режим: {self.current_mode}")
            
            # Если включена имитовставка, проверяем её
            mac_valid = True
            if self.use_mac:
                print(f"\n🛡️  Проверка имитовставки...")
                computed_mac = self.cipher.compute_mac(decrypted, self.mac_len, verbose)
                mac_valid = (computed_mac == expected_mac)
                
                if mac_valid:
                    print(f"   ✅ Имитовставка совпадает! Данные подлинны.")
                else:
                    print(f"   ❌ Имитовставка НЕ совпадает! Данные могли быть изменены.")
                    print(f"      Ожидаемая: {expected_mac.hex()}")
                    print(f"      Вычисленная: {computed_mac.hex()}")
                    
                    # Спрашиваем, сохранять ли данные при несовпадении
                    if input("\nИмитовставка не совпадает. Сохранить данные всё равно? (y/n) [n]: ").lower() != 'y':
                        print("❌ Расшифрование отменено.")
                        input("Нажмите Enter...")
                        return
            
            # Сохраняем результат
            with open(out_file, 'wb') as f:
                f.write(decrypted)
            
            print(f"\n💾 Сохранено {len(decrypted)} байт в {out_file}")
            print(f"   Расшифрованные данные (hex): {decrypted.hex()}")
            
            # Показываем текстовое представление
            try:
                text = decrypted.decode('utf-8')
                print(f"   Текст: {text}")
            except:
                pass
            
            if self.use_mac and not mac_valid:
                print("\n⚠️  ВНИМАНИЕ: Данные сохранены, но имитовставка не совпадает!")
            else:
                print("\n✅ РАСШИФРОВАНИЕ ЗАВЕРШЕНО!")
            
        except Exception as e:
            print(f"\n❌ Ошибка: {e}")
        
        input("\nНажмите Enter...")
    
    def key_menu(self):
        """Меню управления ключами."""
        while True:
            self.clear()
            self.print_header()
            
            print("УПРАВЛЕНИЕ КЛЮЧАМИ")
            print("-" * 70)
            self.print_key_status()
            
            print("  [1] 🔑 Установить ключ из пароля")
            print("  [2] 📂 Загрузить ключ из файла")
            print("  [3] 💾 Сохранить текущий ключ")
            print("  [4] 🎲 Сгенерировать случайный ключ")
            print("  [5] 📋 Показать сохраненные ключи")
            print("  [0] ↩️ Назад")
            print()
            
            choice = input("Выберите действие: ").strip()
            
            if choice == '1':
                self.key_from_password()
            elif choice == '2':
                self.key_from_file()
            elif choice == '3':
                self.key_save()
            elif choice == '4':
                self.key_random()
            elif choice == '5':
                self.list_keys()
            elif choice == '0':
                break
    
    def key_from_password(self):
        """Установка ключа из пароля."""
        print("\n--- Ключ из пароля ---")
        
        pwd = getpass.getpass("Пароль: ")
        pwd2 = getpass.getpass("Подтверждение: ")
        
        if pwd != pwd2:
            print("❌ Пароли не совпадают")
            input("Нажмите Enter...")
            return
        
        print("\n⏳ Генерация ключа...")
        key_bytes = self.cipher.set_key_from_string(pwd)
        self.current_key = key_bytes
        self.current_key_name = f"key_from_password"
        
        print(f"✅ Ключ установлен")
        print(f"   Ключ (hex): {key_bytes.hex()}")
        print(f"   Хеш ключа: {hashlib.sha256(key_bytes).hexdigest()[:16]}...")
        
        if input("\nСохранить ключ в файл? (y/n) [y]: ").lower() != 'n':
            name = input("Имя ключа: ").strip() or f"key_{int(time.time())}"
            path = self.key_manager.save_key(name, key_bytes)
            self.current_key_name = name
            print(f"✅ Ключ сохранен в {path}")
        
        input("Нажмите Enter...")
    
    def key_from_file(self):
        """Загрузка ключа из файла."""
        print("\n--- Загрузка ключа ---")
        
        keys = self.key_manager.list_keys()
        if keys:
            print("\nСохраненные ключи в папке 'keys':")
            for i, key_info in enumerate(keys, 1):
                print(f"  [{i}] {key_info['filename']} ({key_info['modified']})")
            print()
        
        path_input = input("Путь к файлу ключа (или номер): ").strip()
        
        if not path_input:
            print("❌ Путь не указан")
            input("Нажмите Enter...")
            return
        
        if path_input.isdigit() and keys:
            idx = int(path_input) - 1
            if 0 <= idx < len(keys):
                path_input = keys[idx]['path']
                print(f"Выбран ключ: {path_input}")
        
        try:
            key_bytes = self.key_manager.load_key(path_input)
            self.cipher.set_key_from_bytes(key_bytes)
            self.current_key = key_bytes
            self.current_key_name = os.path.basename(path_input)
            
            print(f"\n✅ Ключ загружен из {path_input}")
            print(f"   Ключ (hex): {key_bytes.hex()}")
            print(f"   Хеш ключа: {hashlib.sha256(key_bytes).hexdigest()[:16]}...")
            
        except Exception as e:
            print(f"❌ Ошибка: {e}")
        
        input("Нажмите Enter...")
    
    def key_save(self):
        """Сохранение текущего ключа."""
        if not self.current_key:
            print("❌ Нет ключа для сохранения")
            input("Нажмите Enter...")
            return
        
        print("\n--- Сохранение ключа ---")
        name = input("Имя ключа: ").strip() or f"key_{int(time.time())}"
        
        try:
            path = self.key_manager.save_key(name, self.current_key)
            self.current_key_name = name
            print(f"✅ Ключ сохранен в {path}")
        except Exception as e:
            print(f"❌ Ошибка: {e}")
        
        input("Нажмите Enter...")
    
    def key_random(self):
        """Генерация случайного ключа."""
        print("\n--- Генерация случайного ключа ---")
        
        key_bytes = os.urandom(32)
        self.cipher.set_key_from_bytes(key_bytes)
        self.current_key = key_bytes
        self.current_key_name = f"random_{int(time.time())}"
        
        print(f"✅ Сгенерирован случайный ключ")
        print(f"   Ключ (hex): {key_bytes.hex()}")
        print(f"   Хеш ключа: {hashlib.sha256(key_bytes).hexdigest()[:16]}...")
        
        if input("\nСохранить ключ в файл? (y/n) [y]: ").lower() != 'n':
            self.key_save()
        else:
            input("Нажмите Enter...")
    
    def list_keys(self):
        """Показать список сохраненных ключей."""
        print("\n--- Сохраненные ключи ---")
        keys = self.key_manager.list_keys()
        
        if not keys:
            print("В папке 'keys' нет сохраненных ключей")
        else:
            for i, key_info in enumerate(keys, 1):
                print(f"  [{i}] {key_info['filename']} ({key_info['modified']})")
        
        input("\nНажмите Enter...")
    
    def stats_menu(self):
        """Показать статистику работы."""
        self.clear()
        self.print_header()
        print("📊 СТАТИСТИКА")
        print("-" * 70)
        
        print(f"  Шифрований блоков: {self.cipher.encrypt_count}")
        print(f"  Расшифрований блоков: {self.cipher.decrypt_count}")
        print(f"  Всего операций: {self.cipher.encrypt_count + self.cipher.decrypt_count}")
        
        if self.current_key:
            key_hash = hashlib.sha256(self.current_key).hexdigest()
            print(f"\n  Текущий ключ: {self.current_key_name}")
            print(f"  Хеш ключа: {key_hash[:16]}...")
        
        print(f"  Текущий режим: {self.current_mode if self.current_mode else 'не выбран'}")
        print(f"  Имитовставка: {'ВКЛ' if self.use_mac else 'ВЫКЛ'}")
        if self.use_mac:
            print(f"  Длина MAC: {self.mac_len} байт")
        
        input("\nНажмите Enter...")
    
    def test_menu(self):
        """Тестовый цикл для проверки всех режимов."""
        self.clear()
        self.print_header()
        print("🧪 ТЕСТОВЫЙ ЦИКЛ")
        print("-" * 70)
        
        print("Этот тест проверит все режимы шифрования с опцией имитовставки")
        print("с подробным выводом каждого шага.")
        
        # Спрашиваем, использовать ли имитовставку в тесте
        use_mac_test = input("\nТестировать с имитовставкой? (y/n) [y]: ").lower() != 'n'
        mac_len_test = 8
        if use_mac_test:
            len_input = input("Длина имитовставки (4 или 8) [8]: ").strip()
            if len_input in ('4', '8'):
                mac_len_test = int(len_input)
        
        input("\nНажмите Enter для начала теста...")
        
        # Тестовые данные
        test_data = b"Hello, World! This is a test message."
        print(f"\n📄 Тестовые данные: {test_data.decode('ascii')}")
        print(f"   Размер: {len(test_data)} байт")
        print(f"   Данные (hex): {test_data.hex()}")
        
        # Генерируем тестовый ключ
        test_key = os.urandom(32)
        self.cipher.set_key_from_bytes(test_key)
        print(f"\n🔑 Тестовый ключ: {test_key.hex()}")
        print(f"   Хеш ключа: {hashlib.sha256(test_key).hexdigest()[:16]}...")
        
        # Тестируем все режимы
        test_modes = ["ECB", "CBC", "CFB", "OFB", "CTR"]
        results = []
        
        for mode in test_modes:
            print(f"\n{'='*50}")
            print(f"ТЕСТ РЕЖИМА {mode}")
            print(f"{'='*50}")
            
            # Генерируем IV для режимов, где он нужен
            gamma = os.urandom(16) if mode in ["CBC", "CFB", "OFB", "CTR"] else None
            if gamma:
                print(f"   IV: {gamma.hex()}")
            
            # Шифруем
            print(f"\n🔐 Шифрование {mode}:")
            if mode == "ECB":
                encrypted = self.cipher.encrypt_ecb(test_data, verbose=True)
            elif mode == "CBC":
                encrypted = self.cipher.encrypt_cbc(test_data, gamma, verbose=True)
            elif mode == "CFB":
                encrypted = self.cipher.encrypt_cfb(test_data, gamma, verbose=True)
            elif mode == "OFB":
                encrypted = self.cipher.encrypt_ofb(test_data, gamma, verbose=True)
            elif mode == "CTR":
                encrypted = self.cipher.encrypt_ctr(test_data, gamma, verbose=True)
            
            print(f"\n   🔐 Зашифровано: {len(encrypted)} байт")
            print(f"   Результат: {encrypted.hex()}")
            
            # Если тестируем с имитовставкой, вычисляем и добавляем её
            if use_mac_test:
                print(f"\n🛡️  Вычисление имитовставки...")
                mac = self.cipher.compute_mac(test_data, mac_len_test, verbose=True)
                print(f"   Имитовставка: {mac.hex()}")
                data_with_mac = encrypted + mac
            else:
                data_with_mac = encrypted
            
            # Расшифровываем (с проверкой MAC)
            print(f"\n🔓 Расшифрование {mode}:")
            
            if use_mac_test:
                # Отделяем MAC от данных
                encrypted_for_decrypt = data_with_mac[:-mac_len_test]
                expected_mac = data_with_mac[-mac_len_test:]
                print(f"   Ожидаемая MAC: {expected_mac.hex()}")
            else:
                encrypted_for_decrypt = data_with_mac
            
            if mode == "ECB":
                decrypted = self.cipher.decrypt_ecb(encrypted_for_decrypt, verbose=True)
            elif mode == "CBC":
                decrypted = self.cipher.decrypt_cbc(encrypted_for_decrypt, gamma, verbose=True)
            elif mode == "CFB":
                decrypted = self.cipher.decrypt_cfb(encrypted_for_decrypt, gamma, verbose=True)
            elif mode == "OFB":
                decrypted = self.cipher.decrypt_ofb(encrypted_for_decrypt, gamma, verbose=True)
            elif mode == "CTR":
                decrypted = self.cipher.decrypt_ctr(encrypted_for_decrypt, gamma, verbose=True)
            
            print(f"\n   🔓 Расшифровано: {len(decrypted)} байт")
            print(f"   Результат: {decrypted.hex()}")
            
            # Проверяем расшифрование
            decryption_ok = (decrypted == test_data)
            
            # Если есть имитовставка, проверяем её
            mac_ok = True
            if use_mac_test:
                print(f"\n🛡️  Проверка имитовставки...")
                computed_mac = self.cipher.compute_mac(decrypted, mac_len_test, verbose=True)
                mac_ok = (computed_mac == expected_mac)
                if mac_ok:
                    print(f"   ✅ Имитовставка совпадает")
                else:
                    print(f"   ❌ Имитовставка НЕ совпадает")
                    print(f"      Ожидаемая: {expected_mac.hex()}")
                    print(f"      Вычисленная: {computed_mac.hex()}")
            
            # Общий результат
            if decryption_ok and (not use_mac_test or mac_ok):
                print(f"\n   ✅ УСПЕХ: все проверки пройдены")
                results.append(True)
            else:
                print(f"\n   ❌ ОШИБКА: проверки не пройдены")
                results.append(False)
        
        # Итоги
        print("\n" + "=" * 70)
        print("ИТОГИ ТЕСТИРОВАНИЯ:")
        for i, mode in enumerate(test_modes):
            status = "✅" if results[i] else "❌"
            print(f"  {status} {mode}")
        print("=" * 70)
        
        input("\nНажмите Enter для продолжения...")


def main():
    """Главная функция программы."""
    menu = InteractiveMenu()
    try:
        menu.run()
    except KeyboardInterrupt:
        print("\n\n👋 До свидания!")
    except Exception as e:
        print(f"\n❌ Непредвиденная ошибка: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

🔐 КУЗНЕЧИК - ШИФРОВАНИЕ ПО ГОСТ Р 34.12-2015

❌ Ключ не установлен
🛡️  Имитовставка: ВЫКЛ (длина: 8 байт)

ГЛАВНОЕ МЕНЮ:
  [1] 🔐 Зашифровать файл
  [2] 🔓 Расшифровать файл
  [3] 📋 Выбрать режим шифрования
  [4] 🛡️  Настройки имитовставки (MAC)
  [5] 🔑 Управление ключами
  [6] 🎲 Установить синхропосылку (IV)
  [7] 📊 Статистика
  [8] 🧪 Тестовый цикл
  [0] 🚪 Выход



Выберите действие:  8


🔐 КУЗНЕЧИК - ШИФРОВАНИЕ ПО ГОСТ Р 34.12-2015

🧪 ТЕСТОВЫЙ ЦИКЛ
----------------------------------------------------------------------
Этот тест проверит все режимы шифрования с опцией имитовставки
с подробным выводом каждого шага.



Тестировать с имитовставкой? (y/n) [y]:  y
Длина имитовставки (4 или 8) [8]:  4

Нажмите Enter для начала теста... 



📄 Тестовые данные: Hello, World! This is a test message.
   Размер: 37 байт
   Данные (hex): 48656c6c6f2c20576f726c6421205468697320697320612074657374206d6573736167652e

🔑 Тестовый ключ: 87d594eb6c2a0b6d648388a2c1bc0c3d48260b3e14af9ce22c1b9a05783c1a86
   Хеш ключа: a60e935c1596e346...

ТЕСТ РЕЖИМА ECB

🔐 Шифрование ECB:

📋 ECB Encryption:
   Original data (37 bytes): 48656c6c6f2c20576f726c6421205468697320697320612074657374206d6573736167652e
   Padded data (48 bytes): 48656c6c6f2c20576f726c6421205468697320697320612074657374206d6573736167652e0100000000000000000081
   Block 0: 48656c6c6f2c20576f726c6421205468 -> ad7a18a46db9b5cb24f47479003ea736
   Block 1: 697320697320612074657374206d6573 -> 5d993a9aea44956a4f805663ef6f2fb2
   Block 2: 736167652e0100000000000000000081 -> eeecd514d4f994d80550f1a64c8a949e

   🔐 Зашифровано: 48 байт
   Результат: ad7a18a46db9b5cb24f47479003ea7365d993a9aea44956a4f805663ef6f2fb2eeecd514d4f994d80550f1a64c8a949e

🛡️  Вычисление имитовставки...

🔐 MAC Computation

...

✅ Имитовставка включена



Нажмите Enter... 


🔐 КУЗНЕЧИК - ШИФРОВАНИЕ ПО ГОСТ Р 34.12-2015

🔑 Текущий ключ: key_20260315_172528.key [c0cea68768d34b8e...]
📋 Текущий режим: CFB
🎲 Текущая синхропосылка: 866f9cbcc8e9417e61a63564c842fe62
🛡️  Имитовставка: ВКЛ (длина: 8 байт)

ГЛАВНОЕ МЕНЮ:
  [1] 🔐 Зашифровать файл
  [2] 🔓 Расшифровать файл
  [3] 📋 Выбрать режим шифрования
  [4] 🛡️  Настройки имитовставки (MAC)
  [5] 🔑 Управление ключами
  [6] 🎲 Установить синхропосылку (IV)
  [7] 📊 Статистика
  [8] 🧪 Тестовый цикл
  [0] 🚪 Выход



Выберите действие:  0



👋 До свидания!
